# Imports

In [1]:
import torch
import sys


from torch_geometric.datasets import Planetoid

c:\faculdade\Tabalho-Final-XAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
import random
import torch.nn.functional as F
from torch_geometric.nn import  HANConv
from torch_geometric.datasets import DBLP
import os
import pandas as pd

In [3]:
from itertools import combinations
from tqdm import tqdm
import re
from scipy.stats import spearmanr

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [5]:
import numpy as np
import warnings
import pickle
from pathlib import Path

In [6]:
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import PGExplainer
from torch_geometric.explain.config import ModelConfig

# Dados

In [7]:
def load_dataset(name):

    if name in [
        "Cora",
        "CiteSeer",
        "PubMed"
    ]:

        dataset = Planetoid(
            root=f"data/{name}",
            name=name
        )

        return dataset

    elif name == "DBLP":

        dataset = DBLP(
            root="data/DBLP"
        )

        return dataset

    else:

        raise ValueError(
            f"Dataset {name} não suportado."
        )

DBLP

In [8]:
DBdataset = DBLP(
    root='data/DBLP'
)

Extracting data\DBLP\raw\DBLP_processed.zip
Processing...
Done!


In [41]:
dblp = DBdataset[0]

# manter apenas tipos úteis
keep_nodes = {'author', 'paper', 'term'}

# filtra x_dict
dblp.x_dict = {k: v for k, v in dblp.x_dict.items() if k in keep_nodes}

# filtra edges
dblp.edge_index_dict = {
    k: v for k, v in dblp.edge_index_dict.items()
    if k[0] in keep_nodes and k[2] in keep_nodes
}

print(dblp)

HeteroData(
  x_dict={
    author=[4057, 334],
    paper=[14328, 4231],
    term=[7723, 50],
  },
  edge_index_dict={
    (author, to, paper)=[2, 19645],
    (paper, to, author)=[2, 19645],
    (paper, to, term)=[2, 85810],
    (term, to, paper)=[2, 85810],
  },
  author={
    x=[4057, 334],
    y=[4057],
    train_mask=[4057],
    val_mask=[4057],
    test_mask=[4057],
  },
  paper={ x=[14328, 4231] },
  term={ x=[7723, 50] },
  conference={ num_nodes=20 },
  (author, to, paper)={ edge_index=[2, 19645] },
  (paper, to, author)={ edge_index=[2, 19645] },
  (paper, to, term)={ edge_index=[2, 85810] },
  (paper, to, conference)={ edge_index=[2, 14328] },
  (term, to, paper)={ edge_index=[2, 85810] },
  (conference, to, paper)={ edge_index=[2, 14328] }
)


In [42]:
edge_index_dict = {}

for edge_type, edge_index in dblp.edge_index_dict.items():
    src, _, dst = edge_type

    if src != 'conference' and dst != 'conference':
        edge_index_dict[edge_type] = edge_index

dblp.edge_index_dict = edge_index_dict

In [43]:
node_types = ['author', 'paper', 'term']

edge_types = [
    ('author', 'to', 'paper'),
    ('paper', 'to', 'author'),
    ('paper', 'to', 'term'),
    ('term', 'to', 'paper')
]

metadata = (node_types, edge_types)

In [44]:
dblp = dblp.clone()

# remove conference x se existir
if 'conference' in dblp.x_dict:
    del dblp['conference']

In [30]:
print(dblp.node_types)

['author', 'paper', 'term', 'conference']


In [31]:
print(dblp.edge_types)

[('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('paper', 'to', 'conference'), ('term', 'to', 'paper'), ('conference', 'to', 'paper')]


In [12]:
print(dblp["author"])

{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}


In [32]:
for node_type in dblp.node_types:

    print()
    print(node_type)
    print(dblp[node_type])


author
{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}

paper
{'x': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])}

term
{'x': tensor([[-0.6924, -0.4659,  1.1540,  ...,  0.9178,  0.1995, -0.6360],
        [ 1.2031, -0.4003,  0.0740,  ...,  1.3262, -0.3325,  0.8198],
        [ 0.3748,  0.5731,  0.4802,  ...,  1.1522,  0.6010,

In [33]:
for node_type in dblp.node_types:

    print(node_type)
    print("num_nodes =", dblp[node_type].num_nodes)

    if 'x' in dblp[node_type]:
        print("x.shape =", dblp[node_type].x.shape)

    print()

author
num_nodes = 4057
x.shape = torch.Size([4057, 334])

paper
num_nodes = 14328
x.shape = torch.Size([14328, 4231])

term
num_nodes = 7723
x.shape = torch.Size([7723, 50])

conference
num_nodes = 20



# Modelo

In [18]:
class HAN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, metadata, heads=8):
        super().__init__()

        self.conv = HANConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads
        )

        self.lin = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        x = self.conv(x_dict, edge_index_dict)
        
        out = self.lin(x['author'])
        return out

# Treinamento

## Apoio

In [21]:
def train(model,data,optimizer):
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    criterion = torch.nn.CrossEntropyLoss()

    loss = criterion(out[data['author'].train_mask],
                      data['author'].y[data['author'].train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()

In [22]:
@torch.no_grad()
def test(model,data):
    model.eval()

    out = model(data.x_dict, data.edge_index_dict)
    pred = out.argmax(dim=1)

    acc = (pred[data['author'].test_mask] ==
           data['author'].y[data['author'].test_mask]).float().mean()

    return acc.item()

## Baseline

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = dblp.to(device)

model = HAN(
    in_channels=-1,  # PyG infere automaticamente
    hidden_channels=64,
    out_channels=4,  # DBLP tem 4 classes de autores
    metadata=metadata,
    heads=8
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)

In [46]:
for epoch in range(1, 101):
    loss = train(model,data,optimizer)
    acc = test(model,data)

    print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Acc: {acc:.4f}")

Epoch 001, Loss: 1.3948, Acc: 0.4105
Epoch 002, Loss: 1.3614, Acc: 0.5570
Epoch 003, Loss: 1.3223, Acc: 0.6033
Epoch 004, Loss: 1.2740, Acc: 0.6257
Epoch 005, Loss: 1.2194, Acc: 0.6371
Epoch 006, Loss: 1.1600, Acc: 0.6472
Epoch 007, Loss: 1.0962, Acc: 0.6641
Epoch 008, Loss: 1.0286, Acc: 0.6779
Epoch 009, Loss: 0.9583, Acc: 0.6905
Epoch 010, Loss: 0.8865, Acc: 0.7006
Epoch 011, Loss: 0.8143, Acc: 0.7080
Epoch 012, Loss: 0.7431, Acc: 0.7169
Epoch 013, Loss: 0.6740, Acc: 0.7277
Epoch 014, Loss: 0.6079, Acc: 0.7353
Epoch 015, Loss: 0.5455, Acc: 0.7452
Epoch 016, Loss: 0.4875, Acc: 0.7538
Epoch 017, Loss: 0.4341, Acc: 0.7651
Epoch 018, Loss: 0.3856, Acc: 0.7753
Epoch 019, Loss: 0.3419, Acc: 0.7826
Epoch 020, Loss: 0.3031, Acc: 0.7885
Epoch 021, Loss: 0.2690, Acc: 0.7967
Epoch 022, Loss: 0.2392, Acc: 0.8017
Epoch 023, Loss: 0.2135, Acc: 0.8038
Epoch 024, Loss: 0.1915, Acc: 0.8081
Epoch 025, Loss: 0.1727, Acc: 0.8084
Epoch 026, Loss: 0.1566, Acc: 0.8075
Epoch 027, Loss: 0.1429, Acc: 0.8066
E